# EgoBrain — Temporal Alignment, Synchronization & Dataset Lock

**Purpose:** construct and freeze the auditable common 1-second index linking the already-extracted 512-D visual features, 128-D EEG features, video metadata, and marker-derived action labels.

### Locked outputs
- `all_subjects_synchronized_index.csv`
- `all_subjects_synchronized_labeled_index.csv`
- per-subject synchronization CSVs
- video/visual/EEG timeline audits
- marker taxonomy and clock audits
- manifest + SHA256 checksums

**Do not recompute feature extraction here.** This notebook consumes the extracted modalities and produces the dataset hand-off used by all modeling notebooks.

In [ ]:
# ============================================================
# CELL 1 — CONFIGURATION
# ============================================================

from pathlib import Path
import os, sys, json, math, hashlib, shutil, subprocess, warnings
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import torch

warnings.filterwarnings("ignore")

# -----------------------------
# Existing Kaggle feature inputs
# -----------------------------
VISUAL_ROOT = Path(
    "/kaggle/input/datasets/keerthikumarkj/iris-egobrain-visual-features/uniform"
)

EEG_ROOT_CANDIDATES = [
    Path("/kaggle/input/datasets/keerthikumarkj/iris-eeg-features/eeg_features"),
    Path("/kaggle/input/datasets/keerthikumarkj/iris-eeg-features"),
    Path("/kaggle/input/datasets/keerthikumarkj/iris-egobrain-eeg/eeg_features"),
]

# -----------------------------
# Phase-1 outputs
# -----------------------------
OUT_ROOT = Path("/kaggle/working/egobrain_phase1")
VISUAL_TIMELINE_DIR = OUT_ROOT / "visual_timeline"
EEG_TIMELINE_DIR = OUT_ROOT / "eeg_timeline"
MARKER_DIR = OUT_ROOT / "markers"
ALIGN_DIR = OUT_ROOT / "alignment"
AUDIT_DIR = OUT_ROOT / "audit"
MANIFEST_DIR = OUT_ROOT / "manifests"

for p in [
    OUT_ROOT, VISUAL_TIMELINE_DIR, EEG_TIMELINE_DIR,
    MARKER_DIR, ALIGN_DIR, AUDIT_DIR, MANIFEST_DIR
]:
    p.mkdir(parents=True, exist_ok=True)

SUBJECTS = [f"P{i:04d}" for i in range(1, 41)]

VISUAL_DIM_EXPECTED = 512
EEG_DIM_EXPECTED = 128
EEG_FS_EXPECTED = 256.0
WINDOW_SEC = 1.0

# Stage-1 forensic boundary rule: allow a single 5-ms duration
# discrepancy only when it uniquely explains the observed tensor count.
BOUNDARY_TOLERANCE_SEC = 0.005
EXPECTED_VIDEO_COUNT = 415

# The source_order of GoPro files is part of the Stage-1 row mapping.
VIDEO_EXTENSIONS = {".mp4", ".MP4"}

# ============================================================
# ACTION TAXONOMY
# ============================================================
# 27 semantic task classes observed in the EgoBrain marker stream.
SEMANTIC_TASK_CLASSES = [
    "draw_computer_pictures",
    "draw_pictures",
    "draw_tracing",
    "drink_bitter_gourd_juice",
    "drink_coca_cola",
    "drink_water",
    "eat_snack",
    "eat_spicy_strips",
    "eat_sugers",
    "operate_computer_excel",
    "operate_computer_powerpoint",
    "operate_computer_word",
    "play_cards",
    "play_computer_game_keyboard",
    "play_computer_game_mouse",
    "play_cube",
    "play_phone_game_complex",
    "play_phone_game_simple",
    "play_puzzle",
    "play_toy",
    "read_comic_book",
    "read_research_arcticle",
    "read_textbook",
    "watch_monitor",
    "watch_smartphone",
    "write_notes",
    "write_tracing",
]

# Lifecycle events retained for temporal auditing but included in
# the 29-class action vocabulary because they delimit the experiment.
RETAINED_LIFECYCLE_CLASSES = [
    "finish_running",
    "last_stimuli",
]

EXPECTED_ACTION_CLASS_COUNT = 29

# After canonicalization, the real dataset contains 39 distinct marker
# categories: 29 retained task/lifecycle classes + 10 non-action markers.
# The earlier 31-category gate was incorrect and is now replaced by an
# exact expected-set gate.
EXPECTED_FULL_CANONICAL_CLASSES = set(SEMANTIC_TASK_CLASSES + RETAINED_LIFECYCLE_CLASSES) | {
    "cq",
    "eyesclose_element_q956",
    "eyesopen_element_hcdo",
    "phase_calibration",
    "published_date",
    "sensor_check",
    "sound_check",
    "text_element_k0ik",
    "calibration_phase",
    "calibration_phase_end",
}
EXPECTED_FULL_CANONICAL_CLASS_COUNT = len(EXPECTED_FULL_CANONICAL_CLASSES)

# Canonicalization lowercases unknown/raw marker strings, so CQ becomes `cq`.
NON_ACTION_CANONICAL = {
    "cq",
    "eyesclose_element_q956",
    "eyesopen_element_hcdo",
    "phase_calibration",
    "published_date",
    "sensor_check",
    "sound_check",
    "text_element_k0ik",
    "calibration_phase",
    "calibration_phase_end",
}

# Explicit aliases used by the dataset's marker strings.
CANONICAL_ALIASES = {
    "phase_Calibration_phase": "calibration_phase",
    "phase_Calibration_phase_(End)": "calibration_phase_end",
    "phase_Draw_Computer_Pictures": "draw_computer_pictures",
    "phase_Draw_Pictures": "draw_pictures",
    "phase_Draw_Tracing": "draw_tracing",
    "phase_Drink_Bitter_Gourd_Juice": "drink_bitter_gourd_juice",
    "phase_Drink_Coca-cola_A": "drink_coca_cola",
    "phase_Drink_Coca-cola_B": "drink_coca_cola",
    "phase_Drink_Coca-cola_C": "drink_coca_cola",
    "phase_Drink_Water_A": "drink_water",
    "phase_Drink_Water_B": "drink_water",
    "phase_Drink_Water_C": "drink_water",
    "phase_Eat_Snack_A": "eat_snack",
    "phase_Eat_Snack_B": "eat_snack",
    "phase_Eat_Snack_C": "eat_snack",
    "phase_Eat_Spicy_Strips": "eat_spicy_strips",
    "phase_Eat_Sugers_A": "eat_sugers",
    "phase_Eat_Sugers_B": "eat_sugers",
    "phase_Eat_Sugers_C": "eat_sugers",
    "phase_Operate_Computer_(Excel)": "operate_computer_excel",
    "phase_Operate_Computer_(PowerPoint)": "operate_computer_powerpoint",
    "phase_Operate_Computer_(Word)": "operate_computer_word",
    "phase_Play_Cards": "play_cards",
    "phase_Play_Computer_Game_(Keyboard)": "play_computer_game_keyboard",
    "phase_Play_Computer_Game_(Mouse)": "play_computer_game_mouse",
    "phase_Play_Cube": "play_cube",
    "phase_Play_Phone_Game_(Complex)_A": "play_phone_game_complex",
    "phase_Play_Phone_Game_(Complex)_B": "play_phone_game_complex",
    "phase_Play_Phone_Game_(Simple)_A": "play_phone_game_simple",
    "phase_Play_Phone_Game_(Simple)_B": "play_phone_game_simple",
    "phase_Play_Puzzle": "play_puzzle",
    "phase_Play_Toy": "play_toy",
    "phase_Read_Comic-book": "read_comic_book",
    "phase_Read_Research_Arcticle": "read_research_arcticle",
    "phase_Read_Textbook": "read_textbook",
    "phase_Watch_Monitor": "watch_monitor",
    "phase_Watch_Smartphone": "watch_smartphone",
    "phase_Write_Notes": "write_notes",
    "phase_Write_Tracing": "write_tracing",
    "finish_running": "finish_running",
    "last_stimuli": "last_stimuli",
}

print("Phase 1 configured.")
print("Subjects:", len(SUBJECTS))
print("Expected retained action classes:", EXPECTED_ACTION_CLASS_COUNT)
print("Output:", OUT_ROOT)


In [ ]:
# ============================================================
# CELL 2 — RESOLVE INPUT ROOTS + VALIDATE EXISTING FEATURES
# ============================================================

assert VISUAL_ROOT.exists(), f"Visual root not found: {VISUAL_ROOT}"

EEG_ROOT = next((p for p in EEG_ROOT_CANDIDATES if p.exists()), None)
if EEG_ROOT is None:
    raise FileNotFoundError(
        "Could not locate EEG feature root. Checked:\n" +
        "\n".join(str(p) for p in EEG_ROOT_CANDIDATES)
    )

def find_subject_visual_file(subject):
    candidates = sorted(VISUAL_ROOT.glob(f"*{subject}*.pt"))
    if not candidates:
        candidates = sorted(VISUAL_ROOT.glob(f"{subject}.pt"))
    if not candidates:
        raise FileNotFoundError(f"No visual tensor found for {subject}")
    # Prefer exact feature naming.
    exact = [p for p in candidates if p.name == f"features_{subject}.pt"]
    return exact[0] if exact else candidates[0]

def find_subject_eeg_file(subject):
    candidates = [
        EEG_ROOT / f"eeg_{subject}.pt",
        EEG_ROOT / f"eeg_features_{subject}.pt",
        EEG_ROOT / f"{subject}.pt",
    ]
    for p in candidates:
        if p.exists():
            return p
    globbed = sorted(EEG_ROOT.glob(f"*{subject}*.pt"))
    if globbed:
        return globbed[0]
    raise FileNotFoundError(f"No EEG tensor found for {subject}")

def md5_file(path, chunk=1024*1024):
    h = hashlib.md5()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

visual_rows = []
eeg_rows = []

for subject in SUBJECTS:
    vp = find_subject_visual_file(subject)
    ep = find_subject_eeg_file(subject)

    v = torch.load(vp, map_location="cpu", weights_only=False)
    e = torch.load(ep, map_location="cpu", weights_only=False)

    # Visual artifact may be either a bare tensor or a dict containing features.
    vt = v["features"] if isinstance(v, dict) and "features" in v else v
    et = e["features"] if isinstance(e, dict) and "features" in e else e

    assert isinstance(vt, torch.Tensor), f"{subject}: visual object is not a tensor"
    assert isinstance(et, torch.Tensor), f"{subject}: EEG object is not a tensor"

    assert vt.ndim == 2 and vt.shape[1] == VISUAL_DIM_EXPECTED, (
        f"{subject}: visual shape {tuple(vt.shape)} != [N, {VISUAL_DIM_EXPECTED}]"
    )
    assert et.ndim == 2 and et.shape[1] == EEG_DIM_EXPECTED, (
        f"{subject}: EEG shape {tuple(et.shape)} != [N, {EEG_DIM_EXPECTED}]"
    )

    if isinstance(e, dict):
        fs = float(e.get("sampling_rate_hz", EEG_FS_EXPECTED))
        starts = e.get("time_start_sec")
        ends = e.get("time_end_sec")
    else:
        fs = EEG_FS_EXPECTED
        starts = None
        ends = None

    assert abs(fs - EEG_FS_EXPECTED) < 1e-6, f"{subject}: unexpected EEG fs {fs}"

    if starts is not None and ends is not None:
        starts_np = torch.as_tensor(starts).cpu().numpy()
        ends_np = torch.as_tensor(ends).cpu().numpy()
        assert len(starts_np) == len(et)
        assert len(ends_np) == len(et)
        assert np.allclose(starts_np, np.arange(len(et)), atol=1e-6)
        assert np.allclose(ends_np, np.arange(1, len(et)+1), atol=1e-6)

    visual_rows.append({
        "subject_id": subject,
        "path": str(vp),
        "num_features": int(vt.shape[0]),
        "feature_dim": int(vt.shape[1]),
        "dtype": str(vt.dtype),
        "md5": md5_file(vp),
        "size_mb": vp.stat().st_size / 1024**2,
    })

    eeg_rows.append({
        "subject_id": subject,
        "path": str(ep),
        "num_windows": int(et.shape[0]),
        "feature_dim": int(et.shape[1]),
        "dtype": str(et.dtype),
        "start_sec": 0.0,
        "end_sec": float(et.shape[0]),
        "sampling_rate_hz": fs,
        "num_channels": 32,
        "md5": md5_file(ep),
        "size_mb": ep.stat().st_size / 1024**2,
    })

visual_audit = pd.DataFrame(visual_rows)
eeg_audit = pd.DataFrame(eeg_rows)

print("VISUAL")
display(visual_audit)

print("\nEEG")
display(eeg_audit)

assert len(visual_audit) == 40
assert len(eeg_audit) == 40

visual_audit.to_csv(AUDIT_DIR / "visual_tensor_audit.csv", index=False)
eeg_audit.to_csv(AUDIT_DIR / "eeg_tensor_audit.csv", index=False)

print("\nPASS: all 40 visual tensors and 40 EEG timelines validated.")


## Video timeline reconstruction

Stage 1 produced one 512-D feature for each **complete 1-second video interval**. The current feature counts are therefore expected to equal:

`sum(floor(video_duration_sec))`

over the subject's GoPro files, in the same deterministic source order used by Stage 1.

This is the key recovery step: the old `.pt` tensors do not contain timestamps, but their row-to-video mapping can be reconstructed from the verified video order and durations **without recomputing the visual embeddings**.

The patched metadata cell first reuses an existing `video_metadata.csv`, then checks whether the 415 MP4s are already mounted locally, and only as a last resort accesses the gated Hugging Face files using a Kaggle/Hugging Face token.

We do not use approximate FPS-based frame counts for the feature index; we use the actual video duration metadata and generate only complete one-second intervals.


In [ ]:
# ============================================================
# CELL 3 — FIND / BUILD VIDEO METADATA (PATCHED)
# ============================================================

# Priority:
#   1) Reuse an existing metadata CSV anywhere under /kaggle/input or /kaggle/working.
#   2) If MP4s themselves are mounted locally, measure their durations with ffprobe.
#   3) Only then access the gated Hugging Face files, using a Kaggle secret/token.
#
# IMPORTANT:
# We do NOT download 415 MP4 files unless the first two routes fail.

VIDEO_METADATA_CANDIDATES = [
    Path("/kaggle/input/datasets/keerthikumarkj/egobrain-phase-1-output/egobrain_phase1/video_metadata.csv"),
    Path("/kaggle/working/video_metadata.csv"),
    Path("/kaggle/working/egobrain_video_metadata.csv"),
    Path("/kaggle/input/datasets/keerthikumarkj/egobrain-video-metadata/video_metadata.csv"),
    OUT_ROOT / "video_metadata.csv",
]

def normalize_video_metadata(df):
    rename = {}
    for c in df.columns:
        lc = str(c).strip().lower()
        if lc in {"subject", "subject_id"}:
            rename[c] = "subject_id"
        elif lc in {"source_order", "order", "video_order"}:
            rename[c] = "source_order"
        elif lc in {"remote_path", "path", "video_path"}:
            rename[c] = "remote_path"
        elif lc in {"duration_sec", "duration", "duration_seconds"}:
            rename[c] = "duration_sec"
        elif lc in {"creation_time_raw", "creation_time", "created"}:
            rename[c] = "creation_time_raw"

    df = df.rename(columns=rename)

    required = {"subject_id", "source_order", "remote_path", "duration_sec"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Video metadata missing columns: {sorted(missing)}")

    df = df.copy()
    df["subject_id"] = df["subject_id"].astype(str)
    df["source_order"] = pd.to_numeric(df["source_order"], errors="coerce")
    df["duration_sec"] = pd.to_numeric(df["duration_sec"], errors="coerce")
    df = df.dropna(subset=["source_order", "duration_sec"])
    df["source_order"] = df["source_order"].astype(int)
    df = df.sort_values(["subject_id", "source_order"]).reset_index(drop=True)
    return df

def discover_local_metadata_csvs():
    roots = [
        Path("/kaggle/input"),
        Path("/kaggle/working"),
    ]
    found = []
    for root in roots:
        if root.exists():
            for p in root.rglob("*.csv"):
                name = p.name.lower()
                if "video_metadata" in name or name in {
                    "video_meta.csv",
                    "egobrain_video_metadata.csv",
                }:
                    found.append(p)
    return found

video_meta = None

# ---- 1. Explicit candidates ----
candidate_paths = list(VIDEO_METADATA_CANDIDATES)

# ---- 2. Recursive discovery under Kaggle mounts ----
for p in discover_local_metadata_csvs():
    if p not in candidate_paths:
        candidate_paths.append(p)

for candidate in candidate_paths:
    if candidate.exists():
        try:
            candidate_df = normalize_video_metadata(pd.read_csv(candidate))
            if (
                len(candidate_df) == 415
                and candidate_df["subject_id"].nunique() == 40
            ):
                video_meta = candidate_df
                print("Using existing video metadata:", candidate)
                break
            else:
                print(
                    "Found metadata but rejected incomplete table:",
                    candidate,
                    f"(rows={len(candidate_df)}, subjects={candidate_df['subject_id'].nunique()})"
                )
        except Exception as ex:
            print("Rejected metadata candidate:", candidate, "->", ex)

# ---- 3. Search for locally mounted MP4 files ----
if video_meta is None:
    print("No reusable metadata CSV found. Searching mounted Kaggle files for MP4s...")

    local_mp4s = []
    for root in [Path("/kaggle/input"), Path("/kaggle/working")]:
        if root.exists():
            local_mp4s.extend(root.rglob("*.MP4"))
            local_mp4s.extend(root.rglob("*.mp4"))

    # Keep only EgoBrain GoPro files when identifiable.
    local_mp4s = [
        p for p in local_mp4s
        if "GoPro_HERO12_Ego" in str(p)
    ]

    # Deduplicate.
    local_mp4s = sorted(set(local_mp4s))

    if len(local_mp4s) == 415:
        print("PASS: found all 415 MP4 files locally.")
        ffprobe = shutil.which("ffprobe")
        if ffprobe is None:
            raise RuntimeError("ffprobe is required to measure local MP4 durations.")

        records = []
        for i, local_path in enumerate(local_mp4s, 1):
            # Recover Pxxxx and a deterministic relative source path.
            parts = local_path.parts
            subject_candidates = [x for x in parts if x.startswith("P") and len(x) == 5]
            if not subject_candidates:
                raise RuntimeError(f"Could not recover subject ID from: {local_path}")
            subject = subject_candidates[0]

            # Prefer the dataset-relative path if available.
            marker = "GoPro_HERO12_Ego"
            s = str(local_path).replace("\\", "/")
            idx = s.find(marker)
            if idx >= 0:
                rel_tail = s[idx:]
                remote_path = f"{subject}/{rel_tail}"
            else:
                remote_path = str(local_path)

            cmd = [
                ffprobe, "-v", "error",
                "-show_entries", "format=duration",
                "-of", "default=noprint_wrappers=1:nokey=1",
                str(local_path),
            ]
            duration = float(subprocess.check_output(cmd, text=True).strip())

            records.append({
                "subject_id": subject,
                "remote_path": remote_path,
                "duration_sec": duration,
                "creation_time_raw": None,
            })

            if i % 25 == 0 or i == len(local_mp4s):
                print(f"{i}/{len(local_mp4s)}")

        video_meta = pd.DataFrame(records)

        # Deterministic source order must match the Stage-1 discovery order.
        video_meta["source_order"] = (
            video_meta.sort_values(["subject_id", "remote_path"])
            .groupby("subject_id")
            .cumcount()
        )

        video_meta = normalize_video_metadata(video_meta)

# ---- 4. Last resort: gated Hugging Face access ----
if video_meta is None:
    print("No local metadata/MP4s available. Falling back to Hugging Face.")

    from huggingface_hub import HfApi, hf_hub_download

    HF_REPO = "ut-vision/EgoBrain"
    HF_TYPE = "dataset"

    # Try several common Kaggle secret/environment names.
    HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN")

    try:
        from kaggle_secrets import UserSecretsClient
        secrets = UserSecretsClient()
        for secret_name in ["HF_TOKEN", "HUGGINGFACE_TOKEN", "HUGGINGFACE_API_TOKEN"]:
            if not HF_TOKEN:
                try:
                    HF_TOKEN = secrets.get_secret(secret_name)
                    print(f"Using Kaggle secret: {secret_name}")
                except Exception:
                    pass
    except Exception:
        pass

    if not HF_TOKEN:
        raise RuntimeError(
            "\n"
            "Hugging Face access token required.\n"
            "The EgoBrain dataset is gated and the current notebook has no token.\n\n"
            "FIX:\n"
            "1. Create/verify a Hugging Face access token with access to ut-vision/EgoBrain.\n"
            "2. In Kaggle: Add the token as a secret named HF_TOKEN.\n"
            "3. Re-run this cell.\n\n"
            "Do NOT continue to Phase 1 synchronization until video durations are available.\n"
        )

    api = HfApi(token=HF_TOKEN)

    infos = api.list_repo_tree(
        repo_id=HF_REPO,
        repo_type=HF_TYPE,
        recursive=True,
        expand=False,
    )

    mp4_paths = sorted([
        getattr(item, "path", "")
        for item in infos
        if Path(getattr(item, "path", "")).suffix.lower() in VIDEO_EXTENSIONS
        and "/GoPro_HERO12_Ego/" in getattr(item, "path", "")
    ])

    if len(mp4_paths) != 415:
        raise RuntimeError(
            f"Expected 415 GoPro MP4 files, discovered {len(mp4_paths)}. "
            "Do not continue with an incomplete video inventory."
        )

    print(f"PASS: discovered {len(mp4_paths)} MP4 files.")

    ffprobe = shutil.which("ffprobe")
    if ffprobe is None:
        raise RuntimeError("ffprobe is required when video metadata is not already available.")

    records = []
    tmp_dir = OUT_ROOT / "_video_tmp"
    tmp_dir.mkdir(exist_ok=True)

    try:
        for i, remote_path in enumerate(mp4_paths, 1):
            subject = Path(remote_path).parts[0]
            local_path = None

            try:
                local_path = Path(
                    hf_hub_download(
                        repo_id=HF_REPO,
                        repo_type=HF_TYPE,
                        filename=remote_path,
                        local_dir=str(tmp_dir),
                        token=HF_TOKEN,
                    )
                )

                cmd = [
                    ffprobe, "-v", "error",
                    "-show_entries", "format=duration",
                    "-of", "default=noprint_wrappers=1:nokey=1",
                    str(local_path),
                ]
                duration = float(subprocess.check_output(cmd, text=True).strip())

                records.append({
                    "subject_id": subject,
                    "source_order": 0,
                    "remote_path": remote_path,
                    "duration_sec": duration,
                    "creation_time_raw": None,
                })

                if i % 10 == 0 or i == len(mp4_paths):
                    print(f"{i}/{len(mp4_paths)}")

            finally:
                if local_path is not None:
                    try:
                        local_path.unlink(missing_ok=True)
                    except Exception:
                        pass
    finally:
        shutil.rmtree(tmp_dir, ignore_errors=True)

    video_meta = pd.DataFrame(records)
    video_meta["source_order"] = (
        video_meta.groupby("subject_id").cumcount().astype(int)
    )
    video_meta = normalize_video_metadata(video_meta)

video_meta.to_csv(OUT_ROOT / "video_metadata.csv", index=False)

print("Video metadata rows:", len(video_meta))
print("Subjects:", video_meta["subject_id"].nunique())
display(video_meta.head(20))

assert len(video_meta) == 415
assert video_meta["subject_id"].nunique() == 40
assert video_meta.groupby("subject_id").size().min() >= 1

print("PASS: 415 MP4 files available in deterministic source order.")


In [ ]:
# ============================================================
# CELL 4 — FINAL VISUAL TIMELINE RECONSTRUCTION + FORENSIC GATE
# ============================================================

timeline_records = []
subject_video_audit = []
forensic_rows = []
boundary_corrections = []

for subject in SUBJECTS:
    vrow = visual_audit.loc[visual_audit.subject_id == subject].iloc[0]
    n_visual = int(vrow.num_features)

    vm = (
        video_meta.loc[video_meta.subject_id == subject]
        .sort_values("source_order")
        .reset_index(drop=True)
    )

    if vm.empty:
        raise RuntimeError(f"{subject}: no video metadata")

    durations = vm["duration_sec"].astype(float).to_numpy()
    base_counts = np.floor(durations).astype(int)
    base_total = int(base_counts.sum())
    delta = int(n_visual - base_total)

    # Full-subject forensic statistics used to reproduce the already
    # completed 40-subject audit inside this final notebook.
    forensic_rows.append({
        "subject_id": subject,
        "n_visual": n_visual,
        "n_videos": int(len(vm)),
        "floor_total": base_total,
        "round_total": int(np.round(durations).sum()),
        "ceil_total": int(np.ceil(durations).sum()),
        "tol_0.001_total": int(np.floor(durations + 0.001).sum()),
        "tol_0.005_total": int(np.floor(durations + 0.005).sum()),
        "tol_0.010_total": int(np.floor(durations + 0.010).sum()),
        "diff_floor": delta,
        "exact_duration_sum": float(durations.sum()),
        "near_1ms_files": int(np.sum((durations - np.floor(durations)) >= 0.999)),
        "near_5ms_files": int(np.sum((durations - np.floor(durations)) >= 0.995)),
        "near_10ms_files": int(np.sum((durations - np.floor(durations)) >= 0.990)),
    })

    # Canonical Stage-1 rule: one row per complete 1-second clip.
    clip_counts = base_counts.copy()
    correction_reason = "none"

    if delta == 0:
        pass

    elif delta == 1:
        fractional = durations - np.floor(durations)
        near_boundary = np.flatnonzero(
            fractional >= (1.0 - BOUNDARY_TOLERANCE_SEC)
        )

        # The completed 40-subject forensic audit showed exactly one such
        # video in the entire dataset: the terminal P0016 video at 163.997167 s.
        if len(near_boundary) != 1:
            raise RuntimeError(
                f"{subject}: feature count exceeds floor-duration count by 1, "
                f"but the 5-ms boundary explanation is not unique. "
                f"near_boundary_files={near_boundary.tolist()}"
            )

        idx = int(near_boundary[0])

        # Require the unique boundary file itself to be within 5 ms of the
        # next integer second. This prevents a broad arbitrary rounding rule.
        boundary_error = float(np.ceil(durations[idx]) - durations[idx])
        if boundary_error > BOUNDARY_TOLERANCE_SEC:
            raise RuntimeError(
                f"{subject}: candidate boundary correction is "
                f"{boundary_error*1000:.3f} ms away from the next integer second."
            )

        clip_counts[idx] += 1
        correction_reason = (
            "unique_terminal_duration_boundary_<5ms"
        )

        boundary_corrections.append({
            "subject_id": subject,
            "video_source_order": int(vm.loc[idx, "source_order"]),
            "video_remote_path": vm.loc[idx, "remote_path"],
            "reported_duration_sec": float(durations[idx]),
            "duration_shortfall_to_next_second_sec": boundary_error,
            "added_clip": 1,
            "reason": correction_reason,
        })

    else:
        raise RuntimeError(
            f"{subject}: visual feature count cannot be reconciled with "
            f"Stage-1 duration counts. n_visual={n_visual}, "
            f"floor_total={base_total}, delta={delta}. "
            "Do not proceed to Phase 2."
        )

    reconstructed_total = int(clip_counts.sum())
    if reconstructed_total != n_visual:
        raise RuntimeError(
            f"{subject}: reconstructed clip count {reconstructed_total} "
            f"!= visual tensor rows {n_visual}"
        )

    # Build the row mapping using the recovered number of Stage-1 clips
    # per video. This is the scientifically important part: P0016 gets
    # one additional terminal clip, but no other subject is rounded globally.
    global_feature_idx = 0
    subject_cursor_sec = 0.0

    for row_idx, r in vm.iterrows():
        dur = float(r.duration_sec)
        n_clips = int(clip_counts[row_idx])

        for k in range(n_clips):
            timeline_records.append({
                "subject_id": subject,
                "visual_feature_idx": global_feature_idx,
                "video_source_order": int(r.source_order),
                "video_remote_path": r.remote_path,
                "video_clip_idx": int(k),
                "video_file_start_sec": float(k),
                "video_file_end_sec": float(k + 1),
                "relative_start_sec": float(subject_cursor_sec + k),
                "relative_end_sec": float(subject_cursor_sec + k + 1),
                "duration_sec": dur,
                "stage1_count_method": correction_reason,
            })
            global_feature_idx += 1

        subject_cursor_sec += float(n_clips)

    subject_video_audit.append({
        "subject_id": subject,
        "num_video_files": int(len(vm)),
        "visual_features": n_visual,
        "floor_duration_clips": base_total,
        "reconstructed_stage1_clips": reconstructed_total,
        "difference_before_boundary_correction": delta,
        "video_duration_sum_sec": float(durations.sum()),
        "complete_1s_duration_floor_sum": base_total,
        "boundary_correction_applied": correction_reason != "none",
        "boundary_correction_reason": correction_reason,
    })

# Save both the forensic audit and the final reconstruction audit.
forensic_audit = pd.DataFrame(forensic_rows).sort_values("subject_id")
forensic_audit.to_csv(
    AUDIT_DIR / "all_subject_video_count_forensic_audit.csv",
    index=False
)

video_audit = pd.DataFrame(subject_video_audit).sort_values("subject_id")
video_audit.to_csv(
    AUDIT_DIR / "visual_video_count_audit.csv",
    index=False
)

boundary_audit = pd.DataFrame(boundary_corrections)
if boundary_audit.empty:
    boundary_audit = pd.DataFrame(columns=[
        "subject_id",
        "video_source_order",
        "video_remote_path",
        "reported_duration_sec",
        "duration_shortfall_to_next_second_sec",
        "added_clip",
        "reason",
    ])
boundary_audit.to_csv(
    AUDIT_DIR / "visual_boundary_corrections.csv",
    index=False
)

display(forensic_audit)
display(video_audit)
display(boundary_audit)

# ---------- Forensic global gates ----------
assert len(forensic_audit) == 40
assert int((forensic_audit["n_visual"] == forensic_audit["floor_total"]).sum()) == 39, (
    "Expected exactly 39/40 floor-duration exact matches based on the validated "
    "all-subject forensic audit."
)
assert int((
    forensic_audit["n_visual"]
    == forensic_audit["tol_0.005_total"]
).sum()) == 40, (
    "The validated 5-ms duration tolerance must explain all 40 subject counts."
)

# Exactly one subject should need the boundary correction.
assert len(boundary_audit) == 1
assert boundary_audit.iloc[0]["subject_id"] == "P0016"

# ---------- Reconstruction gates ----------
assert len(video_audit) == 40
assert (video_audit["visual_features"] == video_audit["reconstructed_stage1_clips"]).all()
assert video_audit["boundary_correction_applied"].sum() == 1

visual_timeline = pd.DataFrame(timeline_records)
visual_timeline.to_csv(
    VISUAL_TIMELINE_DIR / "visual_feature_timeline.csv",
    index=False
)

assert len(visual_timeline) == int(visual_audit["num_features"].sum())

for subject, g in visual_timeline.groupby("subject_id"):
    g = g.sort_values("visual_feature_idx").reset_index(drop=True)

    expected_n = int(
        visual_audit.loc[
            visual_audit.subject_id == subject,
            "num_features"
        ].iloc[0]
    )

    assert len(g) == expected_n
    assert g["visual_feature_idx"].is_unique
    assert np.array_equal(
        g["visual_feature_idx"].to_numpy(),
        np.arange(expected_n, dtype=np.int64)
    )
    assert np.allclose(
        g["relative_start_sec"].to_numpy(),
        np.arange(expected_n, dtype=float),
        atol=1e-9
    )
    assert np.allclose(
        g["relative_end_sec"].to_numpy(),
        np.arange(1, expected_n + 1, dtype=float),
        atol=1e-9
    )

print("=" * 78)
print("PASS: 40/40 visual timelines reconstructed.")
print("PASS: 39/40 exact floor-duration matches.")
print("PASS: 40/40 explained by validated 5-ms boundary rule.")
print("PASS: exactly one auditable correction:", boundary_audit.iloc[0]["subject_id"])
print("PASS: visual row → video → 1-second timeline is internally consistent.")
print("=" * 78)


In [ ]:
# ============================================================
# CELL 5 — BUILD EEG TIMELINES FROM EXISTING EEG ARTIFACTS
# ============================================================

eeg_timeline_records = []

for subject in SUBJECTS:
    ep = find_subject_eeg_file(subject)
    e = torch.load(ep, map_location="cpu", weights_only=False)

    et = e["features"] if isinstance(e, dict) and "features" in e else e
    n = int(et.shape[0])

    if isinstance(e, dict):
        fs = float(e.get("sampling_rate_hz", EEG_FS_EXPECTED))
        starts = e.get("time_start_sec")
        ends = e.get("time_end_sec")
    else:
        fs = EEG_FS_EXPECTED
        starts = None
        ends = None

    assert abs(fs - EEG_FS_EXPECTED) < 1e-6

    if starts is not None:
        starts_np = torch.as_tensor(starts).cpu().numpy()
    else:
        starts_np = np.arange(n, dtype=float)

    if ends is not None:
        ends_np = torch.as_tensor(ends).cpu().numpy()
    else:
        ends_np = starts_np + WINDOW_SEC

    assert len(starts_np) == n and len(ends_np) == n

    g = pd.DataFrame({
        "subject_id": subject,
        "eeg_feature_idx": np.arange(n, dtype=np.int64),
        "relative_start_sec": starts_np.astype(float),
        "relative_end_sec": ends_np.astype(float),
    })

    eeg_timeline_records.append(g)

eeg_timeline = pd.concat(eeg_timeline_records, ignore_index=True)
eeg_timeline.to_csv(EEG_TIMELINE_DIR / "eeg_feature_timeline.csv", index=False)

for subject, g in eeg_timeline.groupby("subject_id"):
    g = g.sort_values("eeg_feature_idx")
    assert np.allclose(
        g.relative_start_sec.to_numpy(),
        np.arange(len(g), dtype=float),
        atol=1e-6
    )
    assert np.allclose(
        g.relative_end_sec.to_numpy(),
        np.arange(1, len(g)+1, dtype=float),
        atol=1e-6
    )

print("PASS: EEG timelines validated for all 40 subjects.")
display(
    eeg_timeline.groupby("subject_id")
    .agg(
        eeg_windows=("eeg_feature_idx", "size"),
        eeg_end_sec=("relative_end_sec", "max")
    )
    .reset_index()
)


## Marker timeline

The marker CSV `latency` values are treated as **acquisition-relative seconds**. They are not guessed offsets and are not shifted using semantic keywords.

The JSON clock anchors are used only to audit the absolute acquisition start:

`absolute_marker_timestamp - marker_latency ≈ EEG absolute start`

The expected residual should be very small. The prior audit already found 86–92 clock anchors per subject with sub-microsecond-scale standard deviation, so this is a strong internal consistency check.

For action taxonomy:

- `phase_calibration` / `calibration_phase_end` remain visible in the raw audit.
- They are excluded from the 29 retained action classes.
- `finish_running` and `last_stimuli` are retained as lifecycle classes.
- A/B/C variants of the same task are canonicalized to one class.


In [ ]:
# ============================================================
# CELL 6 — LOAD MARKERS AND BUILD TAXONOMY (AUTH + LOCAL-FIRST FIX)
# ============================================================

# IMPORTANT:
# The EgoBrain dataset is gated. This cell first searches all mounted
# Kaggle input/working locations for already-available intervalMarker.csv
# files. Only missing subjects fall back to Hugging Face.
#
# Authentication is resolved from the Kaggle secret HF_TOKEN first,
# matching the authentication logic that already worked for the video
# metadata stage.

from huggingface_hub import HfApi, hf_hub_download

HF_REPO = "ut-vision/EgoBrain"
HF_TYPE = "dataset"

# ------------------------------------------------------------
# 1. Resolve HF token correctly from Kaggle secret / env
# ------------------------------------------------------------
HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN")

try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    for secret_name in ["HF_TOKEN", "HUGGINGFACE_TOKEN", "HUGGINGFACE_API_TOKEN"]:
        if not HF_TOKEN:
            try:
                candidate = secrets.get_secret(secret_name)
                if candidate:
                    HF_TOKEN = candidate
                    print(f"[AUTH] Using Kaggle secret: {secret_name}")
            except Exception:
                pass
except Exception as ex:
    print("[AUTH] Kaggle secret service unavailable in this runtime:", ex)

# ------------------------------------------------------------
# 2. Local marker discovery — absolutely no HF call if complete
# ------------------------------------------------------------
marker_cache = OUT_ROOT / "_marker_cache"
marker_cache.mkdir(exist_ok=True)

LOCAL_MARKER_ROOTS = [
    Path("/kaggle/working/egobrain_phase0_markers"),
    Path("/kaggle/working/egobrain_markers"),
    Path("/kaggle/input/datasets/keerthikumarkj/egobrain-markers"),
    Path("/kaggle/working"),
    Path("/kaggle/input"),
    marker_cache,
]


def locate_local_marker(subject):
    patterns = [
        f"{subject}*intervalMarker.csv",
        f"*{subject}*intervalMarker.csv",
        f"{subject}*marker*.csv",
        f"*{subject}*marker*.csv",
    ]
    seen = set()
    for root in LOCAL_MARKER_ROOTS:
        if not root.exists():
            continue
        for pat in patterns:
            try:
                found = sorted(root.rglob(pat))
            except Exception:
                found = []
            for p in found:
                if p.is_file() and p not in seen:
                    seen.add(p)
                    # Prefer the actual intervalMarker file over generic marker files.
                    if "intervalmarker" in p.name.lower():
                        return p
                    generic_candidate = p
    # Second pass for generic marker candidates only if necessary.
    for root in LOCAL_MARKER_ROOTS:
        if not root.exists():
            continue
        try:
            found = sorted(root.rglob(f"*{subject}*marker*.csv"))
        except Exception:
            found = []
        for p in found:
            if p.is_file():
                return p
    return None


local_marker_map = {}
for subject in SUBJECTS:
    p = locate_local_marker(subject)
    if p is not None:
        local_marker_map[subject] = p

print(
    f"[MARKERS] Local marker files available: "
    f"{len(local_marker_map)}/{len(SUBJECTS)}"
)

# ------------------------------------------------------------
# 3. Discover remote markers only if required
# ------------------------------------------------------------
missing_subjects = [s for s in SUBJECTS if s not in local_marker_map]
remote_by_subject = defaultdict(list)

if missing_subjects:
    if not HF_TOKEN:
        raise RuntimeError(
            "Marker files are missing locally for "
            f"{missing_subjects}, and no Hugging Face token was found.\n\n"
            "Fix in Kaggle:\n"
            "1. Add your Hugging Face access token as a Kaggle secret named HF_TOKEN.\n"
            "2. Ensure that the token's account has accepted/accessed ut-vision/EgoBrain.\n"
            "3. Re-run this cell.\n"
        )

    print(f"[HF] {len(missing_subjects)} subjects still need remote marker access.")

    api = HfApi(token=HF_TOKEN)

    # Validate authentication explicitly before attempting individual files.
    try:
        who = api.whoami()
        print("[AUTH] Hugging Face authentication OK:", who.get("name", "authenticated"))
    except Exception as ex:
        raise RuntimeError(
            "Hugging Face authentication failed before marker download. "
            "Confirm the Kaggle HF_TOKEN secret is valid and that the account "
            "has access to the gated ut-vision/EgoBrain dataset.\n"
            f"Original error: {type(ex).__name__}: {ex}"
        ) from ex

    tree = api.list_repo_tree(
        repo_id=HF_REPO,
        repo_type=HF_TYPE,
        recursive=True,
        expand=False,
    )

    remote_marker_paths = [
        getattr(item, "path", "")
        for item in tree
        if str(getattr(item, "path", "")).endswith("intervalMarker.csv")
    ]

    for p in remote_marker_paths:
        parts = Path(p).parts
        if parts and parts[0] in SUBJECTS:
            remote_by_subject[parts[0]].append(p)

    print(f"[HF] Remote intervalMarker files discovered: {len(remote_marker_paths)}")

# ------------------------------------------------------------
# 4. Canonicalization
# ------------------------------------------------------------
def canonicalize_marker_type(x):
    s = str(x).strip()
    if s in CANONICAL_ALIASES:
        return CANONICAL_ALIASES[s]

    s2 = (
        s.replace("\\", "")
         .replace(" ", "_")
         .replace("__", "_")
    )

    if s2 in CANONICAL_ALIASES:
        return CANONICAL_ALIASES[s2]

    return s2.lower()

# ------------------------------------------------------------
# 5. Load each subject's marker CSV
# ------------------------------------------------------------
marker_records = []
marker_source_records = []

for subject in SUBJECTS:
    marker_path = local_marker_map.get(subject)

    if marker_path is None:
        candidates = remote_by_subject.get(subject, [])
        if not candidates:
            raise FileNotFoundError(
                f"No marker CSV found locally or remotely for {subject}."
            )

        remote = sorted(candidates)[0]
        marker_path = Path(
            hf_hub_download(
                repo_id=HF_REPO,
                repo_type=HF_TYPE,
                filename=remote,
                local_dir=str(marker_cache),
                token=HF_TOKEN,
            )
        )
        source_kind = "huggingface"
        source_path = remote
    else:
        source_kind = "local"
        source_path = str(marker_path)

    df = pd.read_csv(marker_path)

    required = {"latency", "duration", "type", "marker_value", "timestamp"}
    missing = required - set(df.columns)
    if missing:
        raise RuntimeError(
            f"{subject}: marker CSV missing {sorted(missing)}; source={marker_path}"
        )

    df = df.copy()
    df["subject_id"] = subject
    df["source_row"] = np.arange(len(df), dtype=np.int64)
    df["latency_sec"] = pd.to_numeric(df["latency"], errors="coerce")
    df["duration_sec"] = pd.to_numeric(df["duration"], errors="coerce")
    df["timestamp_unix"] = pd.to_numeric(df["timestamp"], errors="coerce")
    df["raw_type"] = df["type"].astype(str)
    df["canonical_action"] = df["raw_type"].map(canonicalize_marker_type)

    marker_source_records.append({
        "subject_id": subject,
        "source_kind": source_kind,
        "source_path": source_path,
        "rows": int(len(df)),
    })

    marker_records.append(
        df[
            [
                "subject_id", "source_row", "latency_sec", "duration_sec",
                "timestamp_unix", "raw_type", "canonical_action"
            ]
        ]
    )

markers = pd.concat(marker_records, ignore_index=True)
markers = markers.sort_values(
    ["subject_id", "latency_sec", "source_row"]
).reset_index(drop=True)

markers.to_csv(
    MARKER_DIR / "marker_timeline_raw_canonicalized.csv",
    index=False,
)

marker_sources = pd.DataFrame(marker_source_records).sort_values("subject_id")
marker_sources.to_csv(
    AUDIT_DIR / "marker_source_audit.csv",
    index=False,
)

# ------------------------------------------------------------
# 6. Taxonomy audit
# ------------------------------------------------------------
retained_set = set(SEMANTIC_TASK_CLASSES + RETAINED_LIFECYCLE_CLASSES)

full_classes = sorted(
    x for x in markers["canonical_action"].dropna().unique()
    if str(x).strip()
)

taxonomy_rows = []
for c in full_classes:
    g = markers[markers.canonical_action == c]
    taxonomy_rows.append({
        "canonical_action": c,
        "rows": len(g),
        "subjects": g.subject_id.nunique(),
        "is_retained_action": c in retained_set,
        "is_non_action": c in NON_ACTION_CANONICAL,
    })

taxonomy = pd.DataFrame(taxonomy_rows).sort_values("canonical_action")
taxonomy.to_csv(AUDIT_DIR / "marker_taxonomy_audit.csv", index=False)

retained_classes = sorted(retained_set & set(full_classes))

missing_expected = sorted(retained_set - set(retained_classes))
unexpected_retained = sorted(set(retained_classes) - retained_set)

print("Full canonical classes detected:", len(full_classes))
display(taxonomy)

observed_full_set = set(full_classes)
unexpected_full = sorted(observed_full_set - EXPECTED_FULL_CANONICAL_CLASSES)
missing_full = sorted(EXPECTED_FULL_CANONICAL_CLASSES - observed_full_set)
unclassified_full = sorted(observed_full_set - retained_set - set(NON_ACTION_CANONICAL))

print("\nExpected full canonical classes:", EXPECTED_FULL_CANONICAL_CLASS_COUNT)
print("Missing full canonical:", missing_full)
print("Unexpected full canonical:", unexpected_full)
print("Unclassified canonical:", unclassified_full)
print("\nRetained classes:", len(retained_classes))
print("Missing retained:", missing_expected)
print("Unexpected retained:", unexpected_retained)

if observed_full_set != EXPECTED_FULL_CANONICAL_CLASSES:
    raise RuntimeError(
        "FULL MARKER TAXONOMY GATE FAILED: observed canonical marker set "
        "does not exactly match the validated 39-category dataset taxonomy. "
        f"Missing={missing_full}; unexpected={unexpected_full}; "
        f"unclassified={unclassified_full}. "
        "Inspect marker_taxonomy_audit.csv."
    )

if len(retained_classes) != EXPECTED_ACTION_CLASS_COUNT:
    raise RuntimeError(
        f"RETAINED ACTION TAXONOMY GATE FAILED: expected "
        f"{EXPECTED_ACTION_CLASS_COUNT}, detected {len(retained_classes)}. "
        f"Missing={missing_expected}; unexpected={unexpected_retained}"
    )

print(
    "\nPASS: taxonomy is internally consistent: "
    f"{len(full_classes)} full canonical marker categories -> "
    f"{len(retained_classes)} retained action/lifecycle classes."
)


In [ ]:
# ============================================================
# CELL 7 — EEG ABSOLUTE CLOCK AUDIT
# ============================================================

# The first valid marker timestamp/latency pair estimates the absolute
# acquisition start. We use all valid rows and robustly summarize the
# anchor residuals. This is an audit only; EEG feature indexing remains
# relative to 0 seconds.

clock_rows = []

for subject, g in markers.groupby("subject_id"):
    g = g.dropna(subset=["timestamp_unix", "latency_sec"]).copy()
    g["estimated_absolute_start"] = g.timestamp_unix - g.latency_sec

    # Robust central estimate.
    anchor = float(g.estimated_absolute_start.median())
    residual = g.estimated_absolute_start - anchor

    clock_rows.append({
        "subject_id": subject,
        "eeg_absolute_start_unix": anchor,
        "clock_anchor_count": int(len(g)),
        "clock_anchor_std_sec": float(residual.std(ddof=0)),
        "clock_anchor_max_abs_residual_sec": float(residual.abs().max()),
    })

clock_audit = pd.DataFrame(clock_rows).sort_values("subject_id")
clock_audit.to_csv(AUDIT_DIR / "eeg_absolute_clock_audit.csv", index=False)

display(clock_audit)

# The observed dataset has extremely tight marker clock anchors.
# Use a conservative 1 ms gate; this is far looser than the observed values.
if (clock_audit.clock_anchor_std_sec > 1e-3).any():
    bad = clock_audit.loc[
        clock_audit.clock_anchor_std_sec > 1e-3, "subject_id"
    ].tolist()
    raise RuntimeError(f"Clock-anchor consistency failed for {bad}")

print("PASS: absolute marker clock anchors are internally consistent.")


In [ ]:
# ============================================================
# CELL 8 — FINAL CROSS-MODAL SYNCHRONIZED INDEX
# ============================================================

# Both upstream processors construct one-second windows from t=0.
# We therefore use the dataset's common relative recording origin and
# perform an exact interval intersection. We do NOT shift either modality
# using marker semantics.

alignment_records = []
subject_overlap_audit = []

for subject in SUBJECTS:
    vg = (
        visual_timeline[visual_timeline.subject_id == subject]
        .sort_values("visual_feature_idx")
        .reset_index(drop=True)
    )
    eg = (
        eeg_timeline[eeg_timeline.subject_id == subject]
        .sort_values("eeg_feature_idx")
        .reset_index(drop=True)
    )

    if len(vg) == 0 or len(eg) == 0:
        raise RuntimeError(f"{subject}: empty visual or EEG timeline")

    # Exact interval identity:
    # visual [t,t+1) and EEG [t,t+1) share the same relative clock origin.
    visual_start = vg["relative_start_sec"].to_numpy(dtype=float)
    eeg_start = eg["relative_start_sec"].to_numpy(dtype=float)

    common_n = min(len(vg), len(eg))

    if not np.allclose(
        visual_start[:common_n],
        eeg_start[:common_n],
        atol=1e-6,
    ):
        raise RuntimeError(
            f"{subject}: visual/EEG relative clocks do not start at the same "
            "1-second grid. Do not silently offset the modalities."
        )

    pair = pd.DataFrame({
        "subject_id": subject,
        "sync_index": np.arange(common_n, dtype=np.int64),
        "visual_feature_idx": np.arange(common_n, dtype=np.int64),
        "eeg_feature_idx": np.arange(common_n, dtype=np.int64),
    })

    pair["start_sec"] = pair["sync_index"].astype(float)
    pair["end_sec"] = pair["start_sec"] + WINDOW_SEC

    pair["visual_start_sec"] = pair["start_sec"]
    pair["visual_end_sec"] = pair["end_sec"]
    pair["eeg_start_sec"] = pair["start_sec"]
    pair["eeg_end_sec"] = pair["end_sec"]

    pair = pair.merge(
        vg[
            [
                "visual_feature_idx",
                "video_source_order",
                "video_remote_path",
                "video_clip_idx",
                "stage1_count_method",
            ]
        ],
        on="visual_feature_idx",
        how="left",
        validate="one_to_one",
    )

    pair = pair.merge(
        eg[
            [
                "eeg_feature_idx",
                "relative_start_sec",
                "relative_end_sec",
            ]
        ].rename(
            columns={
                "relative_start_sec": "eeg_timeline_start_sec",
                "relative_end_sec": "eeg_timeline_end_sec",
            }
        ),
        on="eeg_feature_idx",
        how="left",
        validate="one_to_one",
    )

    # Explicit common-origin / one-second identity checks.
    assert np.allclose(
        pair["visual_start_sec"],
        pair["eeg_timeline_start_sec"],
        atol=1e-6,
    )
    assert np.allclose(
        pair["visual_end_sec"],
        pair["eeg_timeline_end_sec"],
        atol=1e-6,
    )

    pair.to_csv(
        ALIGN_DIR / f"{subject}_synchronized_index.csv",
        index=False
    )
    alignment_records.append(pair)

    subject_overlap_audit.append({
        "subject_id": subject,
        "visual_windows": int(len(vg)),
        "eeg_windows": int(len(eg)),
        "paired_windows": int(common_n),
        "visual_only_windows": int(max(0, len(vg) - common_n)),
        "eeg_only_windows": int(max(0, len(eg) - common_n)),
        "paired_duration_sec": float(common_n),
        "visual_duration_sec": float(len(vg)),
        "eeg_duration_sec": float(len(eg)),
        "visual_coverage_of_eeg_pct": 100.0 * common_n / len(eg),
        "eeg_coverage_of_visual_pct": 100.0 * common_n / len(vg),
        "common_clock_origin_sec": 0.0,
    })

alignment_index = pd.concat(alignment_records, ignore_index=True)
overlap_audit = pd.DataFrame(subject_overlap_audit).sort_values("subject_id")

alignment_index.to_csv(
    ALIGN_DIR / "all_subjects_synchronized_index.csv",
    index=False
)
overlap_audit.to_csv(
    AUDIT_DIR / "cross_modal_overlap_audit.csv",
    index=False
)

assert len(overlap_audit) == 40
assert len(alignment_index) == int(overlap_audit["paired_windows"].sum())

print("PASS: 40/40 subjects synchronized on the common 1-second relative clock.")
print("Total paired windows:", len(alignment_index))
display(overlap_audit)


In [ ]:
# ============================================================
# CELL 9 — LABEL THE SYNCHRONIZED WINDOWS FROM MARKER INTERVALS
# ============================================================

# Markers are not used to shift the modality clocks.
# They are used to attach experimental context to already-aligned windows.

retained_set = set(SEMANTIC_TASK_CLASSES + RETAINED_LIFECYCLE_CLASSES)

label_records = []

for subject, pair in alignment_index.groupby("subject_id", sort=False):
    pair = pair.copy()

    mg = markers[
        (markers.subject_id == subject) &
        (markers.canonical_action.isin(retained_set))
    ].copy()

    # Marker intervals:
    # [latency, latency + duration)
    # If duration is missing/non-positive, treat it as a point event.
    mg["marker_start_sec"] = mg["latency_sec"].astype(float)
    mg["marker_end_sec"] = (
        mg["latency_sec"].astype(float)
        + mg["duration_sec"].fillna(0).clip(lower=0).astype(float)
    )

    # For each synchronized second, find all marker intervals that overlap it.
    # We keep the raw list first; no arbitrary overwrite is allowed.
    labels = []
    label_conflicts = []

    for _, r in pair.iterrows():
        t0 = float(r.start_sec)
        t1 = float(r.end_sec)

        hit = mg[
            (mg.marker_start_sec < t1) &
            (mg.marker_end_sec > t0)
        ]

        classes = sorted(set(hit.canonical_action.tolist()))

        if len(classes) == 0:
            label = "unlabeled"
            conflict = False
        elif len(classes) == 1:
            label = classes[0]
            conflict = False
        else:
            # Preserve ambiguity instead of silently choosing one.
            label = "ambiguous"
            conflict = True

        labels.append(label)
        label_conflicts.append(conflict)

    pair["action_label"] = labels
    pair["label_conflict"] = label_conflicts

    label_records.append(pair)

labeled_alignment = pd.concat(label_records, ignore_index=True)

labeled_alignment.to_csv(
    ALIGN_DIR / "all_subjects_synchronized_labeled_index.csv",
    index=False
)

label_audit = (
    labeled_alignment.groupby(["subject_id", "action_label"])
    .size()
    .reset_index(name="windows")
    .sort_values(["subject_id", "action_label"])
)

label_audit.to_csv(AUDIT_DIR / "synchronized_label_audit.csv", index=False)

conflicts = labeled_alignment[
    labeled_alignment["label_conflict"]
]

print("Ambiguous synchronized windows:", len(conflicts))
display(label_audit.head(100))

# Ambiguity is allowed for audit, but it must be explicitly quantified.
print(
    f"PASS: label assignment completed. "
    f"Ambiguous windows are retained as 'ambiguous' rather than overwritten: "
    f"{len(conflicts)}"
)


In [ ]:
# ============================================================
# CELL 10 — FINAL RESEARCH-GRADE PHASE-1 GATES
# ============================================================

# ---------- Gate A: subject coverage ----------
assert set(visual_audit.subject_id) == set(SUBJECTS)
assert set(eeg_audit.subject_id) == set(SUBJECTS)
assert set(video_meta.subject_id) == set(SUBJECTS)
assert set(clock_audit.subject_id) == set(SUBJECTS)

# ---------- Gate B: feature dimensions ----------
assert (visual_audit.feature_dim == VISUAL_DIM_EXPECTED).all()
assert (eeg_audit.feature_dim == EEG_DIM_EXPECTED).all()
assert (eeg_audit.sampling_rate_hz == EEG_FS_EXPECTED).all()

# ---------- Gate C: video inventory ----------
assert len(video_meta) == EXPECTED_VIDEO_COUNT
assert video_meta["subject_id"].nunique() == 40

# ---------- Gate D: visual reconstruction ----------
assert len(video_audit) == 40
assert (
    video_audit["visual_features"]
    == video_audit["reconstructed_stage1_clips"]
).all()

# ---------- Gate E: timeline monotonicity ----------
for subject, g in visual_timeline.groupby("subject_id"):
    g = g.sort_values("visual_feature_idx")
    assert g.visual_feature_idx.is_unique
    assert np.all(np.diff(g.relative_start_sec) > 0)

for subject, g in eeg_timeline.groupby("subject_id"):
    g = g.sort_values("eeg_feature_idx")
    assert g.eeg_feature_idx.is_unique
    assert np.all(np.diff(g.relative_start_sec) > 0)

# ---------- Gate F: taxonomy ----------
assert len(full_classes) == EXPECTED_FULL_CANONICAL_CLASS_COUNT == 39
assert set(full_classes) == EXPECTED_FULL_CANONICAL_CLASSES
assert set(retained_classes) == set(
    SEMANTIC_TASK_CLASSES + RETAINED_LIFECYCLE_CLASSES
)
assert len(retained_classes) == EXPECTED_ACTION_CLASS_COUNT == 29
assert set(full_classes) - set(retained_classes) == set(NON_ACTION_CANONICAL)

# ---------- Gate G: clock audit ----------
assert len(clock_audit) == 40
assert (clock_audit.clock_anchor_std_sec <= 1e-3).all()
assert (clock_audit.clock_anchor_max_abs_residual_sec <= 5e-3).all()

# ---------- Gate H: synchronized index ----------
assert labeled_alignment.subject_id.nunique() == 40
assert not labeled_alignment.duplicated(
    ["subject_id", "sync_index"]
).any()

# ---------- Gate I: exact time identity ----------
assert np.allclose(
    labeled_alignment["start_sec"],
    labeled_alignment["visual_start_sec"],
    atol=1e-9,
)
assert np.allclose(
    labeled_alignment["start_sec"],
    labeled_alignment["eeg_timeline_start_sec"],
    atol=1e-6,
)
assert np.allclose(
    labeled_alignment["end_sec"],
    labeled_alignment["eeg_timeline_end_sec"],
    atol=1e-6,
)

print("=" * 82)
print("EgoBrain PHASE 1 — FINAL VALIDATION")
print("=" * 82)
print("PASS  40/40 visual tensors")
print("PASS  40/40 EEG timelines")
print("PASS  415/415 GoPro video metadata rows")
print("PASS  40/40 visual timeline reconstruction")
print("PASS  5-ms boundary forensic gate")
print("PASS  EEG timeline integrity")
print("PASS  marker absolute clock audit")
print("PASS  39 full canonical marker categories (29 retained + 10 non-action)")
print("PASS  29 retained action/lifecycle classes")
print("PASS  cross-modal synchronized 1-second index")
print("PASS  synchronized label audit")
print("=" * 82)
print("PHASE 1 CLEARED")
print("=" * 82)
print()
print("Phase 2 may now use:")
print("  visual tensor + visual_feature_idx")
print("  eeg tensor + eeg_feature_idx")
print("  all_subjects_synchronized_labeled_index.csv")
print()


In [ ]:
# ============================================================
# OPTIONAL PHASE-2 PREFLIGHT — VERIFY THE HAND-OFF
# ============================================================

# This cell does not train anything. It verifies the exact files Phase 2
# will consume and prints a ready-to-use subject-level sample.

phase2_index_path = ALIGN_DIR / "all_subjects_synchronized_labeled_index.csv"

assert phase2_index_path.exists()
assert VISUAL_ROOT.exists()
assert EEG_ROOT.exists()

phase2_index = pd.read_csv(phase2_index_path)

required_cols = {
    "subject_id",
    "visual_feature_idx",
    "eeg_feature_idx",
    "start_sec",
    "end_sec",
    "action_label",
}
missing = required_cols - set(phase2_index.columns)
assert not missing, f"Phase-2 hand-off missing columns: {sorted(missing)}"

assert phase2_index["subject_id"].nunique() == 40
assert not phase2_index.duplicated(
    ["subject_id", "sync_index"]
).any()

demo_subject = SUBJECTS[0]
demo = phase2_index[
    phase2_index.subject_id == demo_subject
].head(10)

print("PHASE-2 PREFLIGHT READY")
print("Index:", phase2_index_path)
print("Total synchronized rows:", len(phase2_index))
print("Subjects:", phase2_index.subject_id.nunique())
print("Example subject:", demo_subject)
display(demo)


In [ ]:
# ============================================================
# CELL 11 — PACKAGE A SINGLE PHASE-1 MANIFEST + CHECKSUMS
# ============================================================

# Compact machine-readable manifest for Phase 2.
phase1_manifest = {
    "phase": "phase_1",
    "status": "CLEARED",
    "phase1_notebook": "EgoBrain_Phase1_FINAL_PHASE2_READY_TAXONOMY_FIXED.ipynb",
    "stage1_visual_count_rule": "floor(duration_sec) except unique <5ms boundary correction validated across 40 subjects",

    "subjects": SUBJECTS,
    "num_subjects": len(SUBJECTS),
    "visual_feature_dim": VISUAL_DIM_EXPECTED,
    "eeg_feature_dim": EEG_DIM_EXPECTED,
    "eeg_sampling_rate_hz": EEG_FS_EXPECTED,
    "window_seconds": WINDOW_SEC,
    "video_files": int(len(video_meta)),
    "full_canonical_marker_classes": int(len(full_classes)),
    "expected_full_canonical_marker_classes": 39,
    "non_action_marker_classes": sorted(NON_ACTION_CANONICAL),
    "retained_action_classes": int(len(retained_classes)),
    "retained_action_class_names": sorted(retained_classes),
    "visual_rows_total": int(len(visual_timeline)),
    "eeg_rows_total": int(len(eeg_timeline)),
    "paired_rows_total": int(len(labeled_alignment)),
    "artifacts": {
        "video_metadata": str(OUT_ROOT / "video_metadata.csv"),
        "visual_timeline": str(VISUAL_TIMELINE_DIR / "visual_feature_timeline.csv"),
        "eeg_timeline": str(EEG_TIMELINE_DIR / "eeg_feature_timeline.csv"),
        "markers": str(MARKER_DIR / "marker_timeline_raw_canonicalized.csv"),
        "taxonomy": str(AUDIT_DIR / "marker_taxonomy_audit.csv"),
        "video_count_forensic": str(AUDIT_DIR / "all_subject_video_count_forensic_audit.csv"),
        "visual_boundary_corrections": str(AUDIT_DIR / "visual_boundary_corrections.csv"),
        "clock_audit": str(AUDIT_DIR / "eeg_absolute_clock_audit.csv"),
        "overlap_audit": str(AUDIT_DIR / "cross_modal_overlap_audit.csv"),
        "alignment_all": str(ALIGN_DIR / "all_subjects_synchronized_index.csv"),
        "alignment_labeled": str(ALIGN_DIR / "all_subjects_synchronized_labeled_index.csv"),
    }
}

with open(MANIFEST_DIR / "phase1_manifest.json", "w") as f:
    json.dump(phase1_manifest, f, indent=2)

# Create a SHA256 manifest over the important generated CSV/JSON artifacts.
artifact_paths = [
    OUT_ROOT / "video_metadata.csv",
    VISUAL_TIMELINE_DIR / "visual_feature_timeline.csv",
    EEG_TIMELINE_DIR / "eeg_feature_timeline.csv",
    MARKER_DIR / "marker_timeline_raw_canonicalized.csv",
    AUDIT_DIR / "marker_taxonomy_audit.csv",
    AUDIT_DIR / "eeg_absolute_clock_audit.csv",
    AUDIT_DIR / "cross_modal_overlap_audit.csv",
    ALIGN_DIR / "all_subjects_synchronized_index.csv",
    ALIGN_DIR / "all_subjects_synchronized_labeled_index.csv",
    MANIFEST_DIR / "phase1_manifest.json",
]

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

checksum_rows = []
for p in artifact_paths:
    checksum_rows.append({
        "artifact": str(p.relative_to(OUT_ROOT)),
        "size_bytes": p.stat().st_size,
        "sha256": sha256_file(p),
    })

checksums = pd.DataFrame(checksum_rows)
checksums.to_csv(MANIFEST_DIR / "phase1_artifact_sha256.csv", index=False)

display(checksums)

print("Manifest written:", MANIFEST_DIR / "phase1_manifest.json")
print("Checksums written:", MANIFEST_DIR / "phase1_artifact_sha256.csv")


In [ ]:
# ============================================================
# CELL 12 — CREATE DOWNLOADABLE ZIP
# ============================================================

# Include the Phase-1 artifacts and per-subject synchronization indices.
# Exclude any temporary raw MP4/marker cache.
ZIP_BASE = Path("/kaggle/working/egobrain_phase1_artifacts")
ZIP_PATH = Path("/kaggle/working/egobrain_phase1_artifacts.zip")

if ZIP_BASE.exists():
    shutil.rmtree(ZIP_BASE)

ZIP_BASE.mkdir(parents=True)

# Copy stable outputs.
for src in [
    OUT_ROOT / "video_metadata.csv",
    VISUAL_TIMELINE_DIR / "visual_feature_timeline.csv",
    EEG_TIMELINE_DIR / "eeg_feature_timeline.csv",
    MARKER_DIR / "marker_timeline_raw_canonicalized.csv",
    AUDIT_DIR,
    MANIFEST_DIR,
    ALIGN_DIR / "all_subjects_synchronized_index.csv",
    ALIGN_DIR / "all_subjects_synchronized_labeled_index.csv",
]:
    if src.is_dir():
        shutil.copytree(src, ZIP_BASE / src.name)
    else:
        shutil.copy2(src, ZIP_BASE / src.name)

# Per-subject alignment files.
subject_dir = ZIP_BASE / "subject_alignment"
subject_dir.mkdir(exist_ok=True)
for p in sorted(ALIGN_DIR.glob("P*_synchronized_index.csv")):
    shutil.copy2(p, subject_dir / p.name)

# Rebuild ZIP.
if ZIP_PATH.exists():
    ZIP_PATH.unlink()

shutil.make_archive(
    base_name=str(ZIP_PATH.with_suffix("")),
    format="zip",
    root_dir=str(ZIP_BASE),
)

zip_mb = ZIP_PATH.stat().st_size / 1024**2

print("=" * 78)
print("PHASE 1 ARTIFACT PACKAGE")
print("=" * 78)
print("Directory:", OUT_ROOT)
print("ZIP:", ZIP_PATH)
print(f"ZIP size: {zip_mb:.2f} MB")
print()
print("IMPORTANT DOWNLOAD FILE:")
print(ZIP_PATH)
print("=" * 78)


## Dataset-lock completion

A successful run must leave the synchronized labeled index, manifests, audits, and checksums under `/kaggle/working/egobrain_phase1` and create the downloadable artifact archive. The locked subject-disjoint split is consumed unchanged by the downstream notebooks.